## Ontology Query Agent — Server-Side Batch Evaluation (Ground Truth)

This notebook evaluates the **deployed** Ontology Query Agent (VKG path, AgentCore Runtime
`QueryRuntimeArn`) **entirely server-side** with Amazon Bedrock **AgentCore Batch
Evaluations** (`bedrock_agentcore.evaluation.BatchEvaluationRunner`). It mirrors
`2_metadata_query_agent_ondemand_groundtruth_eval.ipynb` (the SemanticRAG metadata query
agent) but targets the **VKG** path: the ontology query agent fetches the ontology from
Neptune, disambiguates query terms against it, generates SQL, runs it on Athena, and maps
the result back to RDF N-Quads.

**No scoring runs locally.** The only client-side work is invoking the agent once per
ground-truth row (unavoidable: the agent must run to produce spans) and reading its
response payload to record cost/latency. All evaluator judging happens in-service.

### Why batch (server-side) instead of the on-demand toolkit
The previous revision of this notebook used the on-demand `Evaluation.run` toolkit (which
filters spans by parsing `cloud.resource_id` and was observed to raise *"No spans found"*)
**plus a local `bedrock_runtime.converse` LLM judge**. The **batch** service discovers
spans by **service name + session id + time range** (not `cloud.resource_id`) and runs the
custom judges **in-service**, so neither local runner is needed.

### What gets scored (all server-side)
**Built-in evaluators**

| Evaluator | Level | Ground truth |
|:--|:--|:--|
| `Builtin.GoalSuccessRate` | Session | `assertions` |
| `Builtin.Correctness` | Trace | `expected_response` |

**Custom LLM-as-Judge evaluators** (created with `create_evaluator`, scored in-service — no Lambda)

| Evaluator | Level | Placeholders | Checks |
|:--|:--|:--|:--|
| `QueryAnswerFaithfulness` | TRACE | `{assistant_turn}`, `{expected_response}` | Answer matches the expected answer |
| `SqlGrounded` | SESSION | `{context}`, `{actual_tool_trajectory}` | Every table/column in the executed SQL maps to a class/property in the retrieved ontology (no hallucinated schema) |
| `ToolCallOrdering` | SESSION | `{context}`, `{actual_tool_trajectory}`, `{expected_tool_trajectory}` | Ontology slice retrieved (graph phase) before the SQL Phase 5 ran on Athena |

> **How `SqlGrounded` sees the SQL + the ontology without a Lambda.** The deployed VKG agent
> runs a deterministic Tier 2 graph (phase functions), NOT a ReAct tool loop — the ontology
> fetch, slice assembly, and disambiguation are graph phases, and Phase 5 translates SPARQL→SQL
> (Ontop) and runs it on Athena directly. So there are **no** `get_ontology_from_neptune` /
> `disambiguate_query_terms` / `execute_sql_query` model tool calls. The `{context}` placeholder
> is built from the session spans, whose phase outputs carry the retrieved **ontology slice**
> (classes/properties → tables/columns) and the **executed SQL** (Phase 5 output /
> `reasoning.sqlQuery`). The judge reads both from `{context}` — no code-based/Lambda evaluator
> required.

### Prerequisites
- **Notebook 5** (`5_ontology_agent_ac_eval.ipynb`) run to `%store ontology_id_stored`
  (a completed VKG layer the query agent can resolve), or set `EVAL_ONTOLOGY_ID`
- **AgentCore stack** deployed; Ontology Query Agent running
- `bedrock-agentcore>=1.11.0` and `bedrock-agentcore-starter-toolkit==0.3.9`

In [1]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

## 1. Setup & Credentials

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ AWS Credentials Verified:
  Profile: huthmac+demo-Admin
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1


In [3]:
import re

def _redact_account_ids(obj):
    """Recursively replace AWS account IDs embedded in ARN strings with XXXXXXXXXXXX."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(arn:[^:]*:[^:]*:[^:]*:)\d{12}:', r'\1XXXXXXXXXXXX:', obj)
    return obj


In [4]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# The boto3 bedrock-agentcore client (SigV4) no longer works; use a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 2. Load the Groundtruth Dataset

Schema (one record per row):

| Field | Type | Description |
|:------|:-----|:------------|
| `Natural_Language_Question` | string | The question fed to the agent |
| `Expected_Answer` | string | The natural-language answer the agent should produce |
| `Expected_SQL_Query` | string | The SQL query that would correctly answer the question |
| `Expected_SQL_Result` | list[dict] | The result rows that SQL should return |

The dataset drives the whole run: one scenario per row is built (section 5), the agent is
invoked once per scenario, and the groundtruth fields (`Expected_Answer` →
`expected_response`, the expected SQL/result → `assertions`) feed the server-side built-in
and custom evaluators.

In [5]:
with open('../data/eval/groundtruth_dataset.json', 'r') as f:
    groundtruth = json.load(f)

# Data-query rows need SQL + expected result; advisory/meta rows (Expected_Tier
# == 'advisory') are answered by the intent-router advisory path and carry only
# a question + Expected_Answer, so they are exempt from the SQL requirement.
BASE_COLS = {'Natural_Language_Question', 'Expected_Answer'}
SQL_COLS = {'Expected_SQL_Query', 'Expected_SQL_Result'}
for i, row in enumerate(groundtruth):
    is_advisory = str(row.get('Expected_Tier', '')).lower() == 'advisory'
    required = BASE_COLS if is_advisory else (BASE_COLS | SQL_COLS)
    missing = required - set(row.keys())
    if missing:
        raise ValueError(f"Row {i} missing columns: {missing}")


MAX_ROWS = int(os.environ.get('MAX_ROWS', '0'))
if MAX_ROWS > 0:
    groundtruth = groundtruth[:MAX_ROWS]
    print(f"⚠ MAX_ROWS={MAX_ROWS} — evaluating a {len(groundtruth)}-row slice (set 0 for all)")

df_gt = pd.DataFrame(groundtruth)

# ── Loud ground-truth quality check ──
# The server-side judges + built-in Correctness/GoalSuccessRate score against Expected_Answer /
# Expected_SQL_Query / assertions. Rows whose expected fields are still 'PLACEHOLDER' are scored
# against placeholder ground truth — the numbers are only meaningful once these are filled in.
def _is_placeholder(v) -> bool:
    """True when a ground-truth field is empty or the literal 'PLACEHOLDER' sentinel."""
    return v in (None, '', [], 'PLACEHOLDER')

_ph_sql = sum(1 for r in groundtruth if _is_placeholder(r.get('Expected_SQL_Query')))
_ph_ans = sum(1 for r in groundtruth if _is_placeholder(r.get('Expected_Answer')))
print(f"✓ Loaded {len(df_gt)} groundtruth row(s)")
if _ph_sql or _ph_ans:
    print(f"  ⚠ {_ph_sql}/{len(groundtruth)} rows have a PLACEHOLDER Expected_SQL_Query and "
          f"{_ph_ans}/{len(groundtruth)} have a PLACEHOLDER Expected_Answer.")
    print(f"    Built-in Correctness / GoalSuccessRate and the custom judges will score "
          f"against placeholders for those rows — fill them in for meaningful scores.")
display(df_gt[['Natural_Language_Question', 'Expected_SQL_Query']])

✓ Loaded 16 groundtruth row(s)
  ⚠ 3/16 rows have a PLACEHOLDER Expected_SQL_Query and 0/16 have a PLACEHOLDER Expected_Answer.
    Built-in Correctness / GoalSuccessRate and the custom judges will score against placeholders for those rows — fill them in for meaningful scores.


,Natural_Language_Question,Expected_SQL_Query
0,Show me policies where the insured party is also the policyholder.,"SELECT DISTINCT h.holding_id, h.policy_id, lp.party_id AS insured_party_id\nFROM normalized.holding h\nJOIN normalized.life_participant lp \n ON lp.holding_id = h.holding_id\nJOIN (\n SELECT holding_id, party_id\n FROM normalized.life_participant\n WHERE is_deleted = false\n GROUP BY holding_id, party_id\n HAVING COUNT(DISTINCT participant_sk) > 1\n) dup \n ON dup.holding_id = h.holding_id \n AND dup.party_id = lp.party_id\nWHERE lp.is_deleted = false \n AND h.is_deleted = false"
1,"For each rider, who are the insured participants and what are their roles?","SELECT\n r.rider_id,\n r.rider_name,\n r.rider_code,\n r.rider_type,\n r.rider_status,\n r.holding_id,\n c.coverage_id,\n c.coverage_type AS participant_role,\n c.party_id,\n COALESCE(NULLIF(p.full_name, ''), CONCAT(p.first_name, ' ', p.last_name)) AS participant_name,\n p.party_type,\n c.issue_age,\n c.issue_gender,\n c.underwriting_class\nFROM normalized.rider r\nJOIN normalized.coverage c\n ON r.holding_id = c.holding_id\nLEFT JOIN normalized.party p\n ON c.party_id = p.party_id\nWHERE r.is_deleted = false\n AND c.is_deleted = false\nORDER BY r.rider_id, c.coverage_id\nLIMIT 10"
2,List the top 5 most common party types and their human-readable descriptions.,"SELECT p.party_type, COUNT(*) AS party_count\nFROM normalized.party p\nWHERE p.is_deleted = false\nGROUP BY p.party_type\nORDER BY party_count DESC\nLIMIT 5"
3,"What is the total market value of all active holdings, grouped by party?","SELECT\n p.party_id,\n p.full_name AS party_name,\n COUNT(DISTINCT h.holding_id) AS holding_count,\n SUM(h.market_value) AS total_market_value\nFROM normalized.holding h\nJOIN normalized.coverage c\n ON h.holding_id = c.holding_id\nJOIN normalized.party p\n ON CONCAT('PARTY#', c.party_id) = p.party_id\nWHERE h.holding_status = 'Active'\n AND h.is_deleted = false\n AND c.is_deleted = false\n AND p.is_deleted = false\nGROUP BY p.party_id, p.full_name\nORDER BY total_market_value DESC\nLIMIT 10"
4,"For policies that have a payout schedule, show the policyholder's name, the payout frequency, and the projected next-payout amount.","SELECT\n p.full_name AS policyholder_name,\n COALESCE(pp.premium_mode, c.product_code) AS payout_frequency,\n fa.transaction_amount AS projected_next_payout_amount,\n fa.effective_date AS payout_effective_date,\n fa.activity_type AS payout_type,\n ad.holding_id AS policy_id\nFROM normalized.annuity_detail ad\nJOIN normalized.coverage c\n ON c.holding_id = ad.holding_id\n AND c.is_deleted = false\nJOIN normalized.party p\n ON p.party_id = CONCAT('PARTY#', c.party_id)\n AND p.is_deleted = false\nLEFT JOIN normalized.policy_product pp\n ON pp.product_code = c.product_code\n AND pp.is_deleted = false\nLEFT JOIN normalized.financial_activity fa\n ON fa.holding_id = ad.holding_id\n AND fa.activity_type IN ('Withdrawal', 'Dividend', 'Claim')\n AND fa.is_deleted = false\nWHERE ad.deleted = false\nORDER BY ad.holding_id, fa.effective_date DESC\nLIMIT 10"
5,How many parties are there?,SELECT COUNT(*) AS party_count FROM normalized.party WHERE is_deleted = false
6,List the top 10 coverage products by name.,SELECT product_name FROM normalized.coverage_product WHERE is_deleted = false ORDER BY product_name ASC LIMIT 10
7,"Show the top 10 parties by total holding market value, including the investment product names they hold.","SELECT\n p.party_id,\n p.full_name AS party_name,\n SUM(h.market_value) AS total_market_value,\n ARRAY_JOIN(ARRAY_AGG(DISTINCT\n CASE\n WHEN hs.fundname IS NOT NULL THEN hs.fundname\n WHEN pp.product_name IS NOT NULL THEN pp.product_name\n ELSE 'N/A'\n END\n ), ', ') AS investment_products\nFROM normalized.party p\nJOIN normalized.coverage c\n ON c.party_id = REPLACE(p.party_id, 'PARTY#', '')\n AND c.is_deleted = false\nJOIN normalized.holding h\n ON h.holding_id = c.holding_id\n AND h.is_deleted = false\nLEFT JOIN normalized.holding_suba

## 3. Resolve the AgentCore Ontology Query Runtime

Restore `ontology_id_stored` (stored by notebook 5 — the VKG layer the query agent
resolves) and look up the Ontology Query Runtime ARN (`QueryRuntimeArn`) from the AgentCore
CloudFormation stack outputs.

In [6]:
# ── Resolve the VKG ontology layer to evaluate (best-coverage, validated) ──
# The ground-truth questions span MANY tables (party, policy, holding, coverage,
# rider, ...), so the evaluation ontology must model the full namespace. A
# 1-table smoke-test layer (e.g. from a notebook-5 MAX_TABLES=1 run) cannot answer
# them — the slice judge correctly finds the needed classes missing and the agent
# emits no SQL (0 rows on every row). So:
#   * an explicit EVAL_ONTOLOGY_ID always wins (operator override);
#   * otherwise prefer the completed VKG layer with the MOST dataSources (best
#     ground-truth coverage), NOT merely the newest — a fresh 1-table smoke test
#     must never shadow the full-namespace ontology;
#   * a stored ontology_id_stored (from notebook 5) is honored only if it is the
#     best-coverage layer; otherwise we fall through to it.
_meta_table = session.resource('dynamodb', region_name=region).Table(
    os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata')
)


def _completed_vkg_layers() -> list:
    """Return completed ``vkg-ontology-*`` items with their dataSource counts.

    Returns:
        List of (layer_id, num_tables, updatedAt) tuples, completed layers only.
    """
    items, scan_kw = [], {
        'FilterExpression': 'contains(id, :p) AND #s = :c',
        'ExpressionAttributeNames': {'#s': 'status'},
        'ExpressionAttributeValues': {':p': 'vkg-ontology', ':c': 'completed'},
    }
    while True:
        page = _meta_table.scan(**scan_kw)
        items.extend(page.get('Items', []))
        if 'LastEvaluatedKey' not in page:
            break
        scan_kw['ExclusiveStartKey'] = page['LastEvaluatedKey']
    out = []
    for it in items:
        n = len(it.get('dataSources') or [])
        upd = it.get('updatedAt') or it.get('createdAt') or ''
        out.append((it['id'], n, upd))
    return out


def _best_coverage_vkg() -> str:
    """Return the completed VKG layer id with the most tables (newest breaks ties)."""
    layers = _completed_vkg_layers()
    if not layers:
        return ''
    layers.sort(key=lambda t: (t[1], t[2]), reverse=True)  # (num_tables, updatedAt)
    return layers[0][0]


def _layer_table_count(layer_id: str) -> int:
    """dataSource count for a specific completed layer id, or -1 if absent/incomplete."""
    if not layer_id:
        return -1
    resp = _meta_table.query(
        KeyConditionExpression='id = :i',
        ExpressionAttributeValues={':i': layer_id},
    )
    for it in resp.get('Items', []):
        if it.get('status') == 'completed':
            return len(it.get('dataSources') or [])
    return -1


# An explicit env override always wins; else the stored id from notebook 5.
_explicit = os.environ.get('EVAL_ONTOLOGY_ID', '')
_stored = ''
try:
    %store -r ontology_id_stored
    _stored = ontology_id_stored
except Exception:
    _stored = ''

_best = _best_coverage_vkg()
if not _best and not _explicit and not _stored:
    raise RuntimeError(
        "No completed VKG (vkg-ontology-*) layer found. Run notebook 5 / the admin UI "
        "first, or set EVAL_ONTOLOGY_ID to a completed VKG layer id."
    )

if _explicit and _layer_table_count(_explicit) >= 0:
    EVAL_ID = _explicit
    print(f"✓ Using EVAL_ONTOLOGY_ID override: {EVAL_ID} "
          f"({_layer_table_count(EVAL_ID)} table(s))")
else:
    # Prefer the best-coverage layer. Honor the stored id only if it IS the
    # best-coverage one (so a 1-table smoke test never shadows the full ontology).
    _best_n = _layer_table_count(_best)
    _stored_n = _layer_table_count(_stored)
    if _stored and _stored_n >= _best_n and _stored_n > 0:
        EVAL_ID = _stored
        print(f"✓ Using stored ontology_id (best coverage): {EVAL_ID} ({_stored_n} table(s))")
    else:
        EVAL_ID = _best
        if _stored and _stored_n < _best_n:
            print(f"⚠ Stored ontology_id {_stored!r} has only {_stored_n} table(s) — "
                  f"using the best-coverage VKG layer instead.")
        print(f"✓ Using best-coverage VKG layer: {EVAL_ID} ({_best_n} table(s))")
    ontology_id_stored = EVAL_ID
    %store ontology_id_stored

if _layer_table_count(EVAL_ID) < 2:
    print(f"  ⚠ WARNING: the chosen ontology models only {_layer_table_count(EVAL_ID)} "
          f"table(s); multi-table ground-truth questions will not be answerable. "
          f"Build a full-namespace ontology (notebook 5 with MAX_TABLES=0).")

# ── Look up Ontology Query Runtime ARN + derive agent_id ──
cfn = session.client('cloudformation', region_name=region)
stack_name = f'{PROJECT_NAME}-agentcore'
try:
    outputs = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]['Outputs']
except Exception:
    stack_name = f'{PROJECT_NAME}-dev-agentcore'
    outputs = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]['Outputs']
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}

ONTOLOGY_QUERY_RUNTIME_ARN = output_map['QueryRuntimeArn']
# arn:aws:bedrock-agentcore:region:account:runtime/<agent_id>
agent_id = ONTOLOGY_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"\n✓ AgentCore Runtime (from stack '{stack_name}'):")
print(f"  Runtime ARN: {ONTOLOGY_QUERY_RUNTIME_ARN}")
print(f"  Agent ID:    {agent_id}")

✓ Using stored ontology_id (best coverage): vkg-ontology_curated_layer-8b3529dc (40 table(s))
Stored 'ontology_id_stored' (str)



✓ AgentCore Runtime (from stack 'semantic-layer-dev-agentcore'):
  Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:runtime/semantic_layer_dev_ontology_query-HKGVpkE6XK
  Agent ID:    semantic_layer_dev_ontology_query-HKGVpkE6XK


## 4. Create Custom (LLM-as-Judge) Evaluators — server-side, no Lambda

These run **inside AgentCore Evaluations** as LLM-as-a-Judge evaluators. Each one's
`instructions` reference **placeholders** the service substitutes at evaluation time —
including `{context}`, which carries the **full session conversation** (every tool call and
tool result).

| Evaluator | Level | Placeholders | Checks |
|:--|:--|:--|:--|
| `QueryAnswerFaithfulness` | TRACE | `{assistant_turn}`, `{expected_response}` | Answer matches the expected answer |
| `SqlGrounded` | SESSION | `{context}`, `{actual_tool_trajectory}`, `{available_tools}` | Every table/column in the executed SQL maps to a class/property in the retrieved ontology (no hallucinated schema) |
| `ToolCallOrdering` | SESSION | `{context}`, `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{available_tools}` | Ontology slice retrieved (graph phase) before the SQL Phase 5 ran on Athena |

> **How `SqlGrounded` sees the SQL and the ontology without a Lambda.** The deployed VKG agent
> runs a deterministic Tier 2 graph (phase functions), NOT a tool loop: the ontology fetch /
> slice / disambiguation are graph phases and Phase 5 translates SPARQL→SQL (Ontop) and runs it
> on Athena directly, so there are no `get_ontology_from_neptune` / `disambiguate_query_terms` /
> `execute_sql_query` model tool calls. The `{context}` placeholder is built from the session
> spans, whose phase outputs carry the retrieved **ontology slice** (classes/properties →
> tables/columns) and the **executed SQL** (Phase 5 output / `reasoning.sqlQuery`). The judge
> reads both from `{context}`.

> Re-running this cell creates **new** evaluators (unique name suffix). Delete old ones via
> `delete_evaluator` if you iterate often.

In [7]:
_cp = session.client('bedrock-agentcore-control', region_name=region)
_SUFFIX = uuid.uuid4().hex[:8]
JUDGE_MODEL_ID = 'global.anthropic.claude-sonnet-4-6'

_BINARY_SCALE = {
    'numerical': [
        {'value': 0.0, 'label': 'fail',
         'definition': 'Does not satisfy the criterion.'},
        {'value': 1.0, 'label': 'pass',
         'definition': 'Fully satisfies the criterion.'},
    ]
}


def _create_llm_judge(*, name: str, level: str, instructions: str) -> str:
    """Create a binary LLM-as-Judge evaluator and return its evaluatorId.

    Parameters:
        name: human-readable evaluator name (a unique suffix is appended).
        level: 'TRACE' or 'SESSION' — the granularity the service scores at.
        instructions: judge prompt; may reference ground-truth + context placeholders.

    Returns:
        The created evaluator's evaluatorId.
    """
    resp = _cp.create_evaluator(
        evaluatorName=f"{name}_{_SUFFIX}",
        level=level,
        evaluatorConfig={
            'llmAsAJudge': {
                'instructions': instructions,
                'ratingScale': _BINARY_SCALE,
                'modelConfig': {
                    'bedrockEvaluatorModelConfig': {
                        'modelId': JUDGE_MODEL_ID,
                        'inferenceConfig': {'maxTokens': 1024},
                    }
                },
            }
        },
    )
    return resp['evaluatorId']


# ── Why every custom judge is SESSION-level (no TRACE) — ported from notebook 2 ──
# This VKG agent answers in ONE turn OR asks a clarifying question first and answers on a
# LATER turn (multi-turn, like the RAG agent). SESSION judges read the full {context} across
# all turns and score the FINAL answer once — the right granularity for a multi-turn agent.
# NOTE: clarify turns USED to emit no answer span, so a TRACE evaluator raised "no spans to
# evaluate" and failed the whole session. That is now FIXED IN THE AGENT —
# shared/answer_span.emit_answer_span writes a deterministic span on the clarification and
# final-answer paths, so every turn has an evaluable span. We still use SESSION (not TRACE)
# because answer quality is a per-conversation question and {expected_response} is TRACE-only.
#
# Placeholder constraint (AWS CreateEvaluator): a SESSION evaluator may reference ONLY
# {context}, {available_tools}, {assertions}, {expected_tool_trajectory},
# {actual_tool_trajectory}. {expected_response} is TRACE-ONLY — so the expected final answer
# is threaded in as a SESSION assertion (eval_multiturn.build_trajectory_assertions, prefixed
# FINAL_ANSWER_ASSERTION_PREFIX) and the faithfulness judge reads it from {assertions}.

# SESSION: final-answer faithfulness — the conversation's FINAL answer vs the expected answer
# (carried in {assertions}). Replaces the old TRACE QueryAnswerFaithfulness.
ANSWER_FAITHFUL_ID = _create_llm_judge(
    name='FinalAnswerFaithfulness',
    level='SESSION',
    instructions=(
        "You are a strict binary evaluator for a multi-turn virtual-knowledge-graph (VKG) "
        "text-to-SQL data agent.\n\n"
        "Session context (full conversation across ALL turns — user prompts, assistant "
        "responses, tool calls, and tool results):\n{context}\n\n"
        "Assertions about this session:\n{assertions}\n\n"
        "Exactly one assertion begins with 'The conversation's final answer matches:' — the "
        "text after that prefix is the EXPECTED final answer. The agent may take several "
        "turns (e.g. ask a clarifying question, get a reply, then answer); judge ONLY the "
        "FINAL substantive answer the agent gives at the end of the conversation.\n\n"
        "CRITICAL \u2014 which span is the agent's final answer: this agent runs a "
        "deterministic resolution graph, so each turn emits SEVERAL assistant 'chat' "
        "spans that are NOT the user-facing answer \u2014 in particular an intermediate "
        "SPARQL-GENERATION span (its content is a raw SPARQL query, e.g. 'SELECT "
        "(COUNT(*) AS ?n) ...') and a grounding record (its content begins "
        "'[executed_sql]'). DO NOT treat a SPARQL query, a SQL query, or a "
        "grounding/executed_sql record as the final answer \u2014 those are intermediate "
        "query artifacts, never the answer the user received. The real user-facing "
        "answer is carried in the span whose content is explicitly labelled as the "
        "final-answer record: its INPUT begins with 'Final-answer record for a "
        "deterministic graph turn' and its OUTPUT is the natural-language answer the "
        "user actually saw (e.g. 'The result is 15.', a clarification question, or a "
        "degraded-explanation sentence). Score the OUTPUT of THAT final-answer span "
        "against the expected answer; if multiple final-answer records exist "
        "(multi-turn), use the LAST one. Only if NO final-answer record span exists "
        "should you fall back to the last natural-language assistant message \u2014 and a "
        "bare SPARQL/SQL query is NOT a natural-language answer.\n\n"
        "Score 1 (pass) iff that final answer is factually consistent with the expected "
        "answer — same key numbers, entities, and conclusion. Score 0 (fail) if it "
        "contradicts the expected answer, invents figures, omits the requested result, or if "
        "the conversation never reaches a substantive answer (e.g. ends still asking for "
        "clarification). If no assertion carries the expected-answer prefix, score 0 and note "
        "that the ground truth is missing. Briefly justify your score."
    ),
)

# SESSION: SQL grounding — every table/column in the executed SQL must map to a class/property
# in the retrieved ontology slice. {context} carries the full session: the ontology slice
# (classes/properties with mapsToTable/mapsToColumn) and the executed SQL (Phase 5 / Ontop
# output in reasoning.sqlQuery). No execute_sql_query model tool call exists on this path.
SQL_GROUNDED_ID = _create_llm_judge(
    name='SqlGrounded',
    level='SESSION',
    instructions=(
        "You are a strict binary grounding evaluator for a VKG text-to-SQL data agent.\n\n"
        "Session context (full conversation across ALL turns, including tool calls/results):\n"
        "{context}\n\n"
        "Available tools: {available_tools}\n\n"
        "This VKG agent runs a deterministic resolution graph: it fetches the ontology, "
        "assembles an ontology SLICE, generates SPARQL, then Phase 5 translates that SPARQL to "
        "SQL (Ontop) and runs it on Athena DIRECTLY. The fetch/slice steps are graph phases, "
        "not model tool calls, so do NOT expect get_ontology_from_neptune / "
        "disambiguate_query_terms / execute_sql_query tool CALLS. In the context, locate:\n"
        "  (a) the RETRIEVED ONTOLOGY CONTEXT — the ontology slice (classes/properties with "
        "mapsToTable/mapsToColumn) that is the ONLY schema the agent may use; and\n"
        "  (b) the EXECUTED SQL — surfaced in the Phase 5 output / reasoning.sqlQuery / "
        "executed_sql in the context (the SQL the agent actually ran on Athena).\n\n"
        "IMPORTANT — degraded runs: this agent REFUSES to execute SQL it cannot ground. If "
        "there is NO executed SQL in the context (the agent degraded / asked for clarification "
        "rather than running ungrounded SQL), the grounding invariant was UPHELD — score 1 "
        "(pass) and note 'no SQL executed (grounded refusal)'.\n"
        "Otherwise, score 1 (pass) iff EVERY table, column, and join in the executed SQL maps "
        "to a class/property/mapping in the retrieved ontology context (case-insensitive; "
        "tolerate aliases, quoted vs unquoted identifiers, and SQL builtins such as "
        "COUNT/SUM/DATE_TRUNC — those are not schema). Score 0 (fail) if the executed SQL "
        "references a table/column absent from the ontology context (hallucinated schema), or "
        "if SQL was executed but no ontology context is present. Briefly name the first "
        "offending identifier when you fail it."
    ),
)

# SESSION: ordering invariant — ontology grounded BEFORE SQL execution. The fetch/slice steps
# are graph phases (ontology slice in {context}); judge that the slice precedes the executed
# SQL. A clarification-only conversation (no SQL) has no ordering to violate → pass.
TOOL_ORDERING_ID = _create_llm_judge(
    name='ToolCallOrdering',
    level='SESSION',
    instructions=(
        "You are a strict binary evaluator checking whether a VKG text-to-SQL agent grounded "
        "its query in the retrieved ontology BEFORE executing SQL.\n\n"
        "This agent runs a deterministic resolution graph, not a tool loop. The prescribed "
        "flow for a NEW question is, in order: (1) fetch the ontology (graph phase — appears "
        "as the ontology slice in the context); (2) assemble + disambiguate an ontology SLICE; "
        "(3) generate SPARQL against the slice, then Phase 5 translates it to SQL (Ontop) and "
        "runs it on Athena.\n\n"
        "Available tools: {available_tools}\n"
        "Session context (phase outputs + SQL execution across ALL turns, chronological): "
        "{context}\n\n"
        "Score 1 (pass) iff a retrieved ontology context (the slice) is present and precedes "
        "the SQL execution (a follow-up reusing in-scope ontology and skipping a fresh fetch "
        "is acceptable). ALSO score 1 if the conversation never executes SQL (e.g. ends in a "
        "clarifying question) — there is no ordering violation. Score 0 (fail) if SQL is "
        "executed with no retrieved ontology context before it, or against schema never in "
        "the ontology. Briefly explain the offending ordering when you fail it."
    ),
)

CUSTOM_EVALUATOR_IDS = [ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID, TOOL_ORDERING_ID]
print("✓ Custom server-side evaluators created (all SESSION-level, LLM-as-Judge, no Lambda):")
print(f"  FinalAnswerFaithfulness (SESSION) : {ANSWER_FAITHFUL_ID}")
print(f"  SqlGrounded             (SESSION) : {SQL_GROUNDED_ID}")
print(f"  ToolCallOrdering        (SESSION) : {TOOL_ORDERING_ID}")

✓ Custom server-side evaluators created (all SESSION-level, LLM-as-Judge, no Lambda):
  FinalAnswerFaithfulness (SESSION) : FinalAnswerFaithfulness_f706f60e-63dZll6hQa
  SqlGrounded             (SESSION) : SqlGrounded_f706f60e-dvrMrDFisR
  ToolCallOrdering        (SESSION) : ToolCallOrdering_f706f60e-oF1fAmAWgG


## 5. Build the Dataset (one scenario per row) + agent_invoker

**Per-row scenarios** — each ground-truth question is its own `PredefinedScenario` with its
own session, so we get **per-query** evaluator scores (via `fetch_evaluation_events`) and
clean **per-query** agent token/latency rows.

The `agent_invoker` is the only client-side work: it invokes the synchronous query agent
once per scenario and records the agent's own cost/latency (from `metadata.usage` /
`runtimeMs` and wall-clock) into `AGENT_RUNS`, keyed by the framework session id. Its return
value (`agent_output`) is the answer text the TRACE evaluators score.

In [8]:
# Multi-turn chat-stream invoker + dataset builder (ported from notebook 2).
# The VKG agent's @app.entrypoint dispatches to _chat_stream when the payload carries a
# `turnId` (main.py:1900), reading {message, sessionId, ontologyId, mode}. It reuses ONE
# sessionId across a scenario's turns and keys DDB history off it, so turn N+1 resolves
# turn N's clarification exactly as in production. We parse the AG-UI SSE response and fold
# any clarification option labels into the judge-visible output, recording per-turn
# cost/latency keyed by (session_id, turn_idx).
import time
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput)
from bedrock_agentcore.evaluation.runner.dataset_types import Dataset
try:
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)

# Detect the simulation extra once; gate simulated scenarios on it.
try:
    import strands_evals  # noqa: F401
    SIMULATED_ENABLED = True
except ImportError:
    SIMULATED_ENABLED = False
print(f"simulated mode: {'ENABLED' if SIMULATED_ENABLED else 'DISABLED (scripted only)'}")

AGENT_RUNS = {}              # keyed by run_key(session_id, turn_idx)


def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Drive the VKG agent's CHAT-STREAM path so it reads+persists history itself.

    The SDK reuses one session_id across a scenario's turns; the chat entrypoint keys DDB
    history off the payload sessionId (mode='vkg'), so turn N+1 resolves turn N's
    clarification as in production. We parse the SSE response, fold clarification option
    labels into the judge-visible output, and record per-turn cost/latency keyed by
    (session_id, turn_idx).
    """
    sid = invoker_input.session_id
    # turn index derived from AGENT_RUNS so a notebook re-run (AGENT_RUNS reset above) is safe
    turn_idx = sum(1 for k in AGENT_RUNS if k.startswith(f"{sid}#turn"))
    message = (invoker_input.payload if isinstance(invoker_input.payload, str)
               else json.dumps(invoker_input.payload))
    payload = build_chat_payload(message=message, session_id=sid,
                                 ontology_id=EVAL_ID, turn_idx=turn_idx)
    payload["mode"] = "vkg"  # VKG path (build_chat_payload defaults to semantic-rag)
    start = time.time(); err = None; parsed = {}
    try:
        raw = _invoke_runtime(ONTOLOGY_QUERY_RUNTIME_ARN, sid,
                              json.dumps(payload).encode("utf-8"))
        parsed = parse_chat_stream_sse(raw.decode("utf-8", errors="replace"))
        err = parsed.get("error")
    except Exception as exc:  # noqa: BLE001
        err = str(exc); print(f"  ⚠ [{sid} t{turn_idx}] {err}")
    AGENT_RUNS[run_key(sid, turn_idx)] = {
        "scenario_session": sid, "turn_idx": turn_idx, "message": message,
        "answer": parsed.get("answer", ""), "agent_sql": parsed.get("sql", ""),
        "clarified": bool(parsed.get("clarification")),
        "rows": parsed.get("rows", []), "usage": parsed.get("usage", {}),
        "runtime_ms": parsed.get("runtime_ms"),
        "wall_clock_s": round(time.time() - start, 2), "invoke_error": err,
    }
    state = "clarify" if parsed.get("clarification") else ("answer" if parsed.get("sql") else "none")
    print(f"  {'OK' if err is None else 'XX'} [{sid} t{turn_idx}] "
          f"{AGENT_RUNS[run_key(sid, turn_idx)]['wall_clock_s']}s . {state}")
    return AgentInvokerOutput(agent_output=format_agent_output(parsed) or (err or ""))

_specs = [parse_multiturn_row(r, index=i) for i, r in enumerate(groundtruth)]
dataset = Dataset(scenarios=build_scenarios(
    _specs, ontology_id=EVAL_ID, simulated_enabled=SIMULATED_ENABLED))
EXPECTED_TRAJECTORY = []  # deterministic graph; Phase 5 translates SPARQL→SQL (no model tools)
print(f"✓ Dataset: {len(dataset.scenarios)} scenario(s) "
      f"({sum(1 for s in _specs if len(s['turns'])>1)} multi-turn)")

simulated mode: ENABLED
✓ Dataset: 16 scenario(s) (2 multi-turn)


## 6. Run the Batch Evaluation (server-side)

`run_dataset_evaluation` invokes the agent for each scenario (via `agent_invoker`), waits
`ingestion_delay_seconds` for CloudWatch span ingestion, submits a single
`StartBatchEvaluation` job mixing the **built-in** and **custom** evaluators, and polls to
completion. All scoring happens in-service. Span discovery is by **service name + session id
+ time range**.

In [ ]:
# k-run batch runner (ported from notebook 2): run the WHOLE batch EVAL_K times and AVERAGE
# the per-evaluator aggregate scores in cell 17, so the headline VKG numbers are robust to
# LLM-judge + agent stochasticity. Each run clears + repopulates AGENT_RUNS via agent_invoker
# and snapshots that run's aggregate scores, per-query events, agent_runs, and cost/latency.
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)
try:
    from agents.shared.eval_multiturn import group_runs_by_session
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import group_runs_by_session

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    ingestion_delay_seconds=180,
)

# SESSION-only evaluator set (matches notebook 2). Builtin.Correctness and the old TRACE
# QueryAnswerFaithfulness are intentionally DROPPED: TRACE evaluators error on a
# clarification turn (no model/tool span) and fail the whole session. GoalSuccessRate +
# the 3 SESSION custom judges read the full multi-turn {context}/{assertions}.
ALL_EVALUATOR_IDS = [
    'Builtin.GoalSuccessRate',        # SESSION — assertions (path + final-answer)
    *CUSTOM_EVALUATOR_IDS,            # SESSION — FinalAnswerFaithfulness, SqlGrounded, ToolCallOrdering
]

# Map evaluator id → display name for the per-run aggregate scores.
_EVAL_NAME = {ANSWER_FAITHFUL_ID: 'FinalAnswerFaithfulness',
              SQL_GROUNDED_ID: 'SqlGrounded',
              TOOL_ORDERING_ID: 'ToolCallOrdering'}

# k-run repetition knob (read-only, from notebooks/.env). EVAL_K=1 → single run (old behaviour).
EVAL_K = int(os.environ.get('EVAL_K', '3'))
batch_runner = BatchEvaluationRunner(region=region)


def _scenario_id_for_session(sess):
    """Map a framework session id (scenario_id + '-' + uuid) back to its scenario_id."""
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''


def _aggregate_scores(result) -> dict:
    """Pull the per-evaluator average scores out of a finished batch result.

    Parameters:
        result: the BatchEvaluationRunner result for one run.
    Returns:
        {display_name: average_score|None} over ALL_EVALUATOR_IDS.
    """
    scores = {}
    ev = result.evaluation_results
    if ev is not None:
        for es in (ev.evaluator_summaries or []):
            st = es.statistics
            name = _EVAL_NAME.get(es.evaluator_id, es.evaluator_id)
            scores[name] = (round(st.average_score, 4)
                            if st and st.average_score is not None else None)
    return scores


def _agent_perf_snapshot() -> dict:
    """Summarise the CURRENT AGENT_RUNS into one run's cost/latency totals."""
    lat = [r['wall_clock_s'] for r in AGENT_RUNS.values() if r.get('wall_clock_s') is not None]

    def _usum(key: str) -> int:
        return sum(int((r.get('usage') or {}).get(key) or 0) for r in AGENT_RUNS.values())

    return {'turns': len(AGENT_RUNS),
            'avg_wall_clock_s': round(sum(lat) / len(lat), 2) if lat else None,
            'input_tokens': _usum('inputTokens'), 'output_tokens': _usum('outputTokens'),
            'total_tokens': _usum('totalTokens')}


def _fetch_events(result) -> list:
    """Fetch this run's per-query evaluator events (output stream lags COMPLETED — retry)."""
    for _ in range(6):
        try:
            evs = batch_runner.fetch_evaluation_events(result)
            if evs:
                return evs
        except (LookupError, ValueError):
            pass
        time.sleep(20)
    return []


def _event_rows_for_run(result) -> list:
    """Flatten this run's evaluator events into per-(scenario, evaluator) score rows."""
    grouped = group_runs_by_session(AGENT_RUNS)

    def _f(e, key):
        return e[key] if key in e else (e.get('attributes', {}) or {}).get(key)

    rows = []
    for e in _fetch_events(result):
        sess = _f(e, 'session.id') or _f(e, 'gen_ai.session.id')
        turns = grouped.get(sess, [])
        name = _f(e, 'gen_ai.evaluation.name')
        rows.append({
            'scenario_id': _scenario_id_for_session(sess),
            'question': (turns[0]['message'] if turns else ''),
            'evaluator': _EVAL_NAME.get(name, name),
            'score': _f(e, 'gen_ai.evaluation.score.value'),
            'label': _f(e, 'gen_ai.evaluation.score.label'),
        })
    return rows


def _run_one_batch(run_idx: int) -> dict:
    """Run the SESSION-only batch once (fresh AGENT_RUNS); return a per-run snapshot dict."""
    AGENT_RUNS.clear()  # agent_invoker repopulates this per turn for THIS run
    cfg = BatchEvaluationRunConfig(
        batch_evaluation_name=f"ontquery_gt_batch_r{run_idx}_{uuid.uuid4().hex[:8]}",
        description=f"Server-side SESSION-only ground-truth eval of the ontology query agent, "
                    f"run {run_idx}/{EVAL_K}.",
        evaluator_config=BatchEvaluatorConfig(evaluator_ids=ALL_EVALUATOR_IDS),
        data_source=batch_data_source,
        max_concurrent_scenarios=3,   # synchronous agent, up to 900s/row
        polling_timeout_seconds=3600,
        polling_interval_seconds=30,
    )
    print(f"\n=== Run {run_idx}/{EVAL_K}: {cfg.batch_evaluation_name} ===")
    res = batch_runner.run_dataset_evaluation(
        config=cfg, dataset=dataset, agent_invoker=agent_invoker)
    print(f"  ✓ status: {res.status}")
    if res.agent_invocation_failures:
        print(f"  ⚠ Agent invocation failures: {len(res.agent_invocation_failures)}")
    snap = {'run': run_idx, 'batch_id': res.batch_evaluation_id,
            'batch_arn': res.batch_evaluation_arn, 'status': str(res.status),
            'scores': _aggregate_scores(res), 'agent_perf': _agent_perf_snapshot(),
            'events': _event_rows_for_run(res),
            'agent_runs': list(AGENT_RUNS.values())}
    snap['_result'] = res  # kept in-memory only (not serialised) for the last-run detail cell
    print(f"  run {run_idx} scores: {snap['scores']}")
    return snap


print(f"\nEvaluators : {ALL_EVALUATOR_IDS}")
print(f"Scenarios  : {len(dataset.scenarios)}   ·   EVAL_K = {EVAL_K}")
print("Starting k-run batch evaluation (invokes the agent per scenario, then evaluates server-side)...")

RUNS = []           # per-run snapshots (scores + perf + events + agent_runs); averaged in cell 17
for _i in range(1, EVAL_K + 1):
    RUNS.append(_run_one_batch(_i))
    batch_result = RUNS[-1]['_result']  # keep the LAST run bound for the per-query detail cell

print(f"\n✓ Completed {EVAL_K} run(s)")


## 7. Results — aggregate scores, per-query detail, agent cost/latency

Three layers:
1. **Aggregate per-evaluator scores** — from `batch_result.evaluation_results` (in-service).
2. **Per-query evaluator scores** — `fetch_evaluation_events()` reads one event per
   turn-per-evaluator (score, label, explanation) from the batch output log stream, joined
   back to each question by session id.
3. **Per-query agent cost/latency** — from `AGENT_RUNS` (the agent's own tokens / runtimeMs /
   wall-clock; the batch result never carries these).

In [ ]:
# k-run mean scores + last-run per-query detail + preserved JSON dumps (ported from nb2).
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# ════════════════════════════════════════════════════════════════════════════
# A. k-run MEAN aggregate scores (the headline) — averaged over RUNS from cell 15.
# ════════════════════════════════════════════════════════════════════════════
import statistics as _stats

_EVAL_ORDER = ['Builtin.GoalSuccessRate', 'FinalAnswerFaithfulness',
               'SqlGrounded', 'ToolCallOrdering']


def _mean_std(values: list):
    """Return (mean, std, n) over the non-None numeric values (std=0.0 when n<2)."""
    nums = [v for v in values if isinstance(v, (int, float))]
    if not nums:
        return None, None, 0
    mean = round(sum(nums) / len(nums), 4)
    std = round(_stats.pstdev(nums), 4) if len(nums) > 1 else 0.0
    return mean, std, len(nums)


mean_rows = []
for ev_name in _EVAL_ORDER:
    per_run = [r['scores'].get(ev_name) for r in RUNS]
    mean, std, n = _mean_std(per_run)
    mean_rows.append({'Evaluator': ev_name, 'MeanScore': mean, 'Std': std,
                      'Runs': n, 'PerRun': [r['scores'].get(ev_name) for r in RUNS]})
df_mean = pd.DataFrame(mean_rows)


def _perf_mean(key: str):
    return _mean_std([r['agent_perf'].get(key) for r in RUNS])[0]

agent_perf_mean = {
    'avg_wall_clock_s': _perf_mean('avg_wall_clock_s'),
    'input_tokens': _perf_mean('input_tokens'),
    'output_tokens': _perf_mean('output_tokens'),
    'total_tokens': _perf_mean('total_tokens'),
}

# Per-scenario mean GoalSuccess across runs.
_per_scn = {}
for r in RUNS:
    for e in r['events']:
        if e['evaluator'] == 'Builtin.GoalSuccessRate' and isinstance(e['score'], (int, float)):
            _per_scn.setdefault(e['scenario_id'], []).append(e['score'])
per_scenario_goal_mean = {sid: round(sum(v) / len(v), 4) for sid, v in _per_scn.items()}

# Per-scenario agent detail across all k runs (emitted SQL / mean tokens / mean latency).
# Downstream notebook 9 reads this to build its apples-to-apples matched subset (rows where
# BOTH arms emitted SQL) for this arm WITHOUT re-running this notebook's agent. We key off the
# per-run agent_runs snapshots captured in cell 15, folding multi-turn sessions to their FINAL
# turn (the answer-bearing turn) and averaging tokens/latency over the runs that produced them.
def _scn_id_from_session(sess):
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''
_psa = {}
for r in RUNS:
    by_sess = {}
    for rec in r['agent_runs']:
        by_sess.setdefault(rec['scenario_session'], []).append(rec)
    for sess, recs in by_sess.items():
        recs.sort(key=lambda x: x.get('turn_idx', 0))
        final = recs[-1]
        sid = _scn_id_from_session(sess)
        d = _psa.setdefault(sid, {'sql_runs': 0, 'tokens': [], 'latency': []})
        if final.get('agent_sql'):
            d['sql_runs'] += 1
        d['tokens'].append(int((final.get('usage') or {}).get('totalTokens') or 0))
        d['latency'].append(sum(x.get('wall_clock_s') or 0 for x in recs))
# question text per scenario (the robust cross-notebook join key — gt-row-NN indices differ
# between notebooks because some filter the dataset, but the question text is stable).
_q_by_sid = {s['scenario_id']: (s['turns'][0].get('input', '') if s.get('turns') else '')
             for s in _specs}
per_scenario_agent = {
    sid: {'question': _q_by_sid.get(sid, ''),
          'emitted_sql': d['sql_runs'] > 0,        # emitted SQL in at least one run
          'sql_run_fraction': round(d['sql_runs'] / len(d['tokens']), 4) if d['tokens'] else 0.0,
          'avg_total_tokens': round(sum(d['tokens']) / len(d['tokens'])) if d['tokens'] else 0,
          'avg_wall_clock_s': round(sum(d['latency']) / len(d['latency']), 2) if d['latency'] else None}
    for sid, d in _psa.items()
}


print(f"=== k-run MEAN scores over EVAL_K={EVAL_K} run(s) ===")
display(df_mean)
print(f"\nMean agent cost/latency: avg_wall={agent_perf_mean['avg_wall_clock_s']}s  "
      f"total_tokens={agent_perf_mean['total_tokens']}")

# ── Persist the k-run mean summary (the file downstream notebooks 10 & 11 read). ──
kmean_file = f"../data/eval/results/ontology_query_kmean_eval_{timestamp}.json"
kmean = {
    'notebook': '6_ontology_query',
    'arm_label': 'vkg',                    # VKG path; for nb11 this is the WITH-OntologyRAG arm
    'agent': 'ontology_query_agent',
    'agent_id': agent_id,
    'runtime_arn': ONTOLOGY_QUERY_RUNTIME_ARN,
    'eval_id': EVAL_ID,
    'eval_k': EVAL_K,
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'custom_evaluators': {'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                          'SqlGrounded': SQL_GROUNDED_ID, 'ToolCallOrdering': TOOL_ORDERING_ID},
    'mean_scores': {row['Evaluator']: row['MeanScore'] for row in mean_rows},
    'std_scores': {row['Evaluator']: row['Std'] for row in mean_rows},
    'per_run_scores': [{'run': r['run'], 'batch_id': r['batch_id'],
                        'status': r['status'], 'scores': r['scores'],
                        'agent_perf': r['agent_perf']} for r in RUNS],
    'agent_perf_mean': agent_perf_mean,
    'per_scenario_goal_mean': per_scenario_goal_mean,
    'per_scenario_agent': per_scenario_agent,
}
kmean = _redact_account_ids(kmean)
with open(kmean_file, 'w') as f:
    json.dump(kmean, f, indent=2, default=str)
print(f"\n✓ Wrote k-run mean summary → {kmean_file}")

# ════════════════════════════════════════════════════════════════════════════
# B. Per-query DETAIL for the LAST run (batch_result + AGENT_RUNS point at run k).
#    Preserves the ontology_query_batch_eval_*.json file (downstream-compatible).
# ════════════════════════════════════════════════════════════════════════════
try:
    from agents.shared.eval_multiturn import group_runs_by_session
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import group_runs_by_session

print(f"\n── Last-run detail — Batch ID: {batch_result.batch_evaluation_id} "
      f"(status {batch_result.status}) ──")

# ── 1. Aggregate per-evaluator scores (last run) ─────────────────────────────
agg_rows = []
ev = batch_result.evaluation_results
if ev is not None:
    print(f"Sessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (f"{stats.average_score:.3f}"
               if stats and stats.average_score is not None else None)
        agg_rows.append({'Evaluator': es.evaluator_id or 'unknown', 'AvgScore': avg,
                         'Evaluated': es.total_evaluated or 0, 'Failed': es.total_failed or 0})
else:
    print("⚠ No aggregate evaluation_results — check job status / spans.")
df_agg = pd.DataFrame(agg_rows)

grouped = group_runs_by_session(AGENT_RUNS)

# ── 2. Per-query evaluator events (last run) — reuse the helper from cell 15 ──
event_rows = _event_rows_for_run(batch_result)
def _f(e, key):
    return e[key] if key in e else (e.get('attributes', {}) or {}).get(key)
_explan = {}
for e in _fetch_events(batch_result):
    sess = _f(e, 'session.id') or _f(e, 'gen_ai.session.id')
    name = _f(e, 'gen_ai.evaluation.name')
    _explan[(_scenario_id_for_session(sess), _EVAL_NAME.get(name, name))] = \
        (str(_f(e, 'gen_ai.evaluation.explanation') or ''))[:400]
for row in event_rows:
    row['explanation'] = _explan.get((row['scenario_id'], row['evaluator']), '')
df_events = pd.DataFrame(event_rows)
print(f"Per-query evaluator events (last run): {len(df_events)}")

# ── 3. Per-turn trajectory table + clarification summary (last run) ──────────
turn_rows = []
for sess, turns in grouped.items():
    sid = _scenario_id_for_session(sess)
    for t in turns:
        turn_rows.append({
            'scenario_id': sid, 'turn': t.get('turn_idx'),
            'message': (t.get('message') or ''), 'clarified': t.get('clarified'),
            'has_sql': bool(t.get('agent_sql')), 'wall_s': t.get('wall_clock_s'),
            'error': t.get('invoke_error'),
        })
df_agent = pd.DataFrame(turn_rows)
if not df_agent.empty:
    df_agent = df_agent.sort_values(['scenario_id', 'turn']).reset_index(drop=True)

print("\n── Trajectory summary (last run, per scenario) ──")
for sess, turns in grouped.items():
    sid = _scenario_id_for_session(sess)
    clar = [t.get('turn_idx') for t in turns if t.get('clarified')]
    final_sql = bool(turns[-1].get('agent_sql')) if turns else False
    print(f"  {sid}: {len(turns)} turn(s) · clarified_on={clar or 'none'} · "
          f"re_clarified={len(clar) > 1} · final_sql={final_sql}")

# ── 4. Agent cost/latency (last run) ─────────────────────────────────────────
_lat = [r['wall_clock_s'] for r in AGENT_RUNS.values() if r.get('wall_clock_s') is not None]
def _usage_sum(key):
    return sum(int((r.get('usage') or {}).get(key) or 0) for r in AGENT_RUNS.values())
agent_perf = {
    'turns': len(AGENT_RUNS), 'sessions': len(grouped),
    'avg_wall_clock_s': round(sum(_lat) / len(_lat), 2) if _lat else None,
    'input_tokens': _usage_sum('inputTokens'), 'output_tokens': _usage_sum('outputTokens'),
    'total_tokens': _usage_sum('totalTokens'),
}
print(f"\n── Query agent cost/latency (last run) ──")
print(f"  Avg wall-clock : {agent_perf['avg_wall_clock_s']}s  ·  tokens: {agent_perf['total_tokens']}")

# ── Persist the last-run detail. Keep the ontology_query_batch_eval_ prefix so notebook 4
#    (which globs query_batch_eval_*) does NOT pick up the VKG agent's results. ──
combined_file = f"../data/eval/results/ontology_query_batch_eval_{timestamp}.json"
combined = {
    'agent_id': agent_id, 'runtime_arn': ONTOLOGY_QUERY_RUNTIME_ARN, 'eval_id': EVAL_ID,
    'eval_k': EVAL_K, 'kmean_file': os.path.basename(kmean_file),
    'batch_evaluation_id': batch_result.batch_evaluation_id,
    'batch_evaluation_arn': batch_result.batch_evaluation_arn,
    'status': str(batch_result.status), 'evaluator_ids': ALL_EVALUATOR_IDS,
    'custom_evaluators': {'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                          'SqlGrounded': SQL_GROUNDED_ID, 'ToolCallOrdering': TOOL_ORDERING_ID},
    'aggregate_summaries': agg_rows, 'per_query_events': event_rows,
    'agent_runs': list(AGENT_RUNS.values()), 'agent_performance': agent_perf,
}
combined = _redact_account_ids(combined)
with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"✓ Wrote last-run detail → {combined_file}")

print("\n=== Aggregate per-evaluator scores (last run) ===")
display(df_agg)
print("=== Per-query evaluator scores (last run) ===")
display(df_events)
print("=== Per-turn trajectory (last run) ===")
display(df_agent)


## 8. Store IDs for Downstream Notebooks

Save the batch evaluation id, the per-query session ids, and the agent id so downstream
notebooks can reference this run.

In [11]:
ontology_query_batch_id = batch_result.batch_evaluation_id
ontology_query_eval_session_ids = list(AGENT_RUNS.keys())
ondemand_query_agent_id = agent_id           # kept name for downstream notebook compatibility
ondemand_query_session_id = (ontology_query_eval_session_ids[0]
                             if ontology_query_eval_session_ids else None)
%store ontology_query_batch_id
%store ontology_query_eval_session_ids
%store ondemand_query_agent_id
%store ondemand_query_session_id

print("✓ Stored for downstream notebooks:")
print(f"  ontology_query_batch_id         = {ontology_query_batch_id}")
print(f"  ontology_query_eval_session_ids = {len(ontology_query_eval_session_ids)} session(s)")
print(f"  ondemand_query_agent_id         = {ondemand_query_agent_id}")
print(f"\n  Combined results: {combined_file}")

Stored 'ontology_query_batch_id' (str)
Stored 'ontology_query_eval_session_ids' (list)
Stored 'ondemand_query_agent_id' (str)
Stored 'ondemand_query_session_id' (str)
✓ Stored for downstream notebooks:
  ontology_query_batch_id         = ontquery_gt_batch_ca7168a7-44b6abb13c
  ontology_query_eval_session_ids = 19 session(s)
  ondemand_query_agent_id         = semantic_layer_dev_ontology_query-HKGVpkE6XK

  Combined results: ../data/eval/results/ontology_query_batch_eval_20260614_192942.json


## Summary

This notebook evaluates the deployed **Ontology Query Agent** (VKG path) **entirely
server-side** via AgentCore Batch Evaluations. The only client-side work is invoking the
agent once per ground-truth row (required to produce spans) and reading its response for
cost/latency.

### Pipeline
1. **Custom evaluators** (LLM-as-Judge, no Lambda) — `QueryAnswerFaithfulness` (TRACE),
   `SqlGrounded` (SESSION), and `ToolCallOrdering` (SESSION), created with `create_evaluator`
   and scored in-service.
2. **Dataset** — one `PredefinedScenario` per ground-truth row (per-query sessions), each
   carrying `expected_response`, `expected_trajectory`, and `assertions`.
3. **agent_invoker** — invokes the synchronous agent per scenario and records the agent's own
   tokens / `runtimeMs` / wall-clock into `AGENT_RUNS`.
4. **BatchEvaluationRunner** — one `StartBatchEvaluation` job over built-in + custom evaluators.
5. **Results** — aggregate per-evaluator scores, **per-query** scores via
   `fetch_evaluation_events`, and **per-query** agent cost/latency.

### What batch can / can't give you
- ✅ Built-in + custom evaluator scores, server-side (no local scoring).
- ✅ **SQL grounding** and **tool-call ordering** scored server-side: the `{context}` placeholder
  exposes the session's tool calls/results (the executed SQL and the retrieved ontology), so the
  judges read them directly — no code-based/Lambda evaluator required.
- ✅ Per-query evaluator scores (via the output log stream) and per-query agent token/latency.
- ❌ The agent's own tokens are **not** in the batch result — captured by the invoker instead.

### Ground-truth caveat
`data/eval/groundtruth_dataset.json` currently has mostly `PLACEHOLDER` expected SQL/answers.
Correctness, GoalSuccessRate, and `QueryAnswerFaithfulness` only produce meaningful scores once
those rows are filled in (the load cell prints how many are still placeholders). `SqlGrounded`
and `ToolCallOrdering` do **not** depend on the expected SQL/answer — they score against the
agent's own retrieved ontology context + trajectory — so they are meaningful even before the
dataset is completed.

### Next steps
- Fill in `Expected_SQL_Query` / `Expected_Answer` / `Expected_SQL_Result` for all rows.
- Run the comparison notebooks (7–9) for cross-path / cross-source evaluation.